# Transfer Learning with AlexNet — STL-10 Image Classification

This notebook fine-tunes a pretrained **AlexNet** (ImageNet weights) on the
**STL-10** dataset (10 classes, 96x96 native images, auto-downloaded by
`torchvision`) and visualizes the dataset, training progress, and results.

Structured block-by-block:
1. Setup & imports
2. Configuration
3. Load dataset + **visualize sample images**
4. Data augmentation & DataLoaders
5. Load pretrained AlexNet & modify the classifier head
6. Loss, optimizer, scheduler
7. Training loop
8. **Visualize** training curves (loss & accuracy)
9. Evaluate on test set + **confusion matrix**
10. **Visualize predictions** (correct vs incorrect samples)
11. Per-class accuracy chart


## Block 1 — Setup & Imports

In [ ]:
# If running in Colab / a fresh environment, uncomment:
# !pip install torch torchvision matplotlib scikit-learn seaborn -q

import os
import time
import random
import copy

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix, classification_report

print("PyTorch version:", torch.__version__)


## Block 2 — Configuration & Reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", DEVICE)

# --- Hyperparameters ---
IMG_SIZE     = 224          # AlexNet's expected input size
BATCH_SIZE   = 32
EPOCHS       = 10
LR           = 1e-3
MODE         = "finetune"   # "feature_extract" (freeze backbone) or "finetune" (unfreeze last block too)
DATA_DIR     = "./data"
OUTPUT_DIR   = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


## Block 3 — Load the Dataset & Visualize Raw Samples

**Dataset: STL-10** — 10 classes (airplane, bird, car, cat, deer, dog, horse,
monkey, ship, truck), 96x96 native resolution, auto-downloaded here.
(Swap this block for `datasets.ImageFolder(...)` to use your own data.)

In [ ]:
# Load without heavy transforms first, just to visualize raw samples
raw_view_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

raw_train = datasets.STL10(root=DATA_DIR, split="train", download=True, transform=raw_view_tf)
class_names = raw_train.classes
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Training samples (raw): {len(raw_train)}")


In [ ]:
# Visualize a grid of sample images with their labels
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
indices = random.sample(range(len(raw_train)), 18)

for ax, idx in zip(axes.flat, indices):
    img, label = raw_train[idx]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(class_names[label], fontsize=10)
    ax.axis("off")

fig.suptitle("Sample Images from STL-10 Training Set", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_sample_images.png"), dpi=150)
plt.show()


In [ ]:
# Class distribution bar chart
labels = [raw_train.labels[i] for i in range(len(raw_train))]
counts = np.bincount(labels)

plt.figure(figsize=(9, 4))
plt.bar(class_names, counts, color="steelblue")
plt.title("Class Distribution — STL-10 Training Set")
plt.ylabel("Number of images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_class_distribution.png"), dpi=150)
plt.show()


## Block 4 — Data Augmentation & DataLoaders

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# STL-10 ships with predefined 'train' (5000 imgs) and 'test' (8000 imgs) splits.
full_train = datasets.STL10(root=DATA_DIR, split="train", download=True, transform=train_transform)
test_set   = datasets.STL10(root=DATA_DIR, split="test",  download=True, transform=eval_transform)

# Carve out a validation set from the training split
n_val   = int(0.15 * len(full_train))
n_train = len(full_train) - n_val
train_set, val_subset = random_split(
    full_train, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)
# Validation should use eval-time transforms (no augmentation)
val_base = datasets.STL10(root=DATA_DIR, split="train", download=False, transform=eval_transform)
val_set = torch.utils.data.Subset(val_base, val_subset.indices)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")


## Block 5 — Load Pretrained AlexNet & Modify the Classifier Head

- `feature_extract` mode: freeze all convolutional layers, train only the new head.
- `finetune` mode: also unfreeze the last few convolutional layers so the
  network can adapt its higher-level features to this dataset.

In [ ]:
def build_alexnet(num_classes, mode="finetune"):
    weights = models.AlexNet_Weights.IMAGENET1K_V1
    model = models.alexnet(weights=weights)

    # Replace the final classification layer (was 4096 -> 1000 for ImageNet)
    in_feats = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_feats, num_classes)

    # Freeze everything first
    for p in model.parameters():
        p.requires_grad = False
    # Always train the new head
    for p in model.classifier[6].parameters():
        p.requires_grad = True

    if mode == "finetune":
        # Unfreeze the last convolutional block for adaptation
        for p in model.features[-3:].parameters():
            p.requires_grad = True

    return model


model = build_alexnet(num_classes=len(class_names), mode=MODE).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params/1e6:.2f}M | Trainable params: {trainable_params/1e6:.2f}M")
print(model.classifier)


## Block 6 — Loss Function, Optimizer, Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(EPOCHS // 3, 1), gamma=0.1)


## Block 7 — Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    running_loss, running_correct, n_samples = 0.0, 0, 0

    torch.set_grad_enabled(train)
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        if train:
            optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        if train:
            loss.backward()
            optimizer.step()

        preds = outputs.argmax(dim=1)
        running_loss += loss.item() * inputs.size(0)
        running_correct += (preds == labels).sum().item()
        n_samples += inputs.size(0)

    return running_loss / n_samples, running_correct / n_samples


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_state = copy.deepcopy(model.state_dict())

start_time = time.perf_counter()
for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, DEVICE, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, DEVICE, train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} "
          f"| val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

train_time = time.perf_counter() - start_time
model.load_state_dict(best_state)
print(f"\nTraining complete in {train_time:.1f}s | Best val accuracy: {best_val_acc:.4f}")


## Block 8 — Visualize Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"], marker="o", label="Val Loss")
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], marker="o", label="Train Accuracy")
axes[1].plot(epochs_range, history["val_acc"], marker="o", label="Val Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_training_curves.png"), dpi=150)
plt.show()


## Block 9 — Evaluate on Test Set & Confusion Matrix

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_inputs = [], [], []
    for inputs, labels in loader:
        inputs_d = inputs.to(device)
        outputs = model(inputs_d)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
        all_inputs.append(inputs)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy(), torch.cat(all_inputs)

preds, labels, test_inputs = evaluate(model, test_loader, DEVICE)
test_acc = (preds == labels).mean()
print(f"Test Accuracy: {test_acc:.4f}")
print("\nClassification Report:\n")
print(classification_report(labels, preds, target_names=class_names, zero_division=0))


In [ ]:
cm = confusion_matrix(labels, preds)

plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix — AlexNet (Test Accuracy: {test_acc:.4f})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_confusion_matrix.png"), dpi=150)
plt.show()


## Block 10 — Visualize Predictions (Correct vs Incorrect)

In [ ]:
def denormalize(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)

correct_idx = np.where(preds == labels)[0]
incorrect_idx = np.where(preds != labels)[0]

def show_prediction_grid(indices, title, n=8, color="green"):
    n = min(n, len(indices))
    if n == 0:
        print(f"No samples for: {title}")
        return
    chosen = np.random.choice(indices, size=n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, idx in zip(axes, chosen):
        img = denormalize(test_inputs[idx]).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(f"T:{class_names[labels[idx]]}\nP:{class_names[preds[idx]]}",
                     fontsize=9, color=color)
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    fname = "05_correct_predictions.png" if color == "green" else "06_incorrect_predictions.png"
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150)
    plt.show()

show_prediction_grid(correct_idx, "Correct Predictions", n=8, color="green")
show_prediction_grid(incorrect_idx, "Incorrect Predictions", n=8, color="red")


## Block 11 — Per-Class Accuracy Chart

In [ ]:
per_class_acc = []
for i in range(len(class_names)):
    mask = labels == i
    acc_i = (preds[mask] == labels[mask]).mean() if mask.sum() > 0 else 0.0
    per_class_acc.append(acc_i)

plt.figure(figsize=(9, 5))
bars = plt.bar(class_names, per_class_acc, color="teal")
plt.axhline(test_acc, color="red", linestyle="--", label=f"Overall Accuracy = {test_acc:.3f}")
plt.title("Per-Class Test Accuracy — AlexNet")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_per_class_accuracy.png"), dpi=150)
plt.show()

print(f"\nFinal Summary — AlexNet on STL-10")
print(f"Best Val Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy:     {test_acc:.4f}")
print(f"Training Time:     {train_time:.1f}s")
print(f"Trainable Params:  {trainable_params/1e6:.2f}M / {total_params/1e6:.2f}M total")
